In [ ]:
import pandas as pd
import numpy as np

!pip install catboost

from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

# -------------------------
# Load Data
# -------------------------

df = pd.read_csv("givenData.csv")

# -------------------------
# Convert datetime columns
# -------------------------

datetime_cols = [
    "start_datetime",
    "resolved_datetime",
    "closed_datetime"
]

for col in datetime_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# -------------------------
# Create Target
# -------------------------

df["event_end"] = df["resolved_datetime"]

mask = df["event_end"].isna()
df.loc[mask, "event_end"] = df.loc[mask, "closed_datetime"]

df["duration_minutes"] = (
    df["event_end"] - df["start_datetime"]
).dt.total_seconds() / 60

# Remove invalid durations

df = df[
    (df["duration_minutes"] > 0)
    & (df["duration_minutes"] < 10080)
]

# -------------------------
# Time Features
# -------------------------

df["hour"] = df["start_datetime"].dt.hour

df["day_of_week"] = df["start_datetime"].dt.dayofweek

df["month"] = df["start_datetime"].dt.month

df["is_weekend"] = (
    df["day_of_week"] >= 5
).astype(int)

# -------------------------
# Historical Junction Average
# -------------------------

junction_avg = (
    df.groupby("junction")["duration_minutes"]
      .mean()
      .to_dict()
)

df["junction_avg_duration"] = (
    df["junction"]
    .map(junction_avg)
)


# -------------------------
# Feature Selection
# -------------------------

features = [
    "event_type",
    "event_cause",
    "requires_road_closure",
    "priority",
    "veh_type",
    "corridor",
    "police_station",
    "zone",
    "junction",
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
    "junction_avg_duration"
]

X = df[features]
y = df["duration_minutes"]

# -------------------------
# Train Test Split
# -------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# -------------------------
# Categorical Columns
# -------------------------

cat_features = [
    "event_type",
    "event_cause",
    "requires_road_closure",
    "priority",
    "veh_type",
    "corridor",
    "police_station",
    "zone",
    "junction"
]

# Fill NaN values in categorical features with 'Unknown'
for col in cat_features:
    if col in X_train.columns:
        X_train[col] = X_train[col].fillna('Unknown')
    if col in X_test.columns:
        X_test[col] = X_test[col].fillna('Unknown')

# -------------------------
# Model
# -------------------------

model = CatBoostRegressor(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="MAE",
    verbose=100
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_features
)

# -------------------------
# Prediction
# -------------------------

preds = model.predict(X_test)

# -------------------------
# Metrics
# -------------------------

mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)

print("MAE:", round(mae, 2), "minutes")
print("R2:", round(r2, 3))

# -------------------------
# Save Model and Junction Stats
# -------------------------
joblib.dump(model, "duration_model.pkl")
joblib.dump(junction_avg, "junction_stats.pkl")
print("Model and junction statistics saved.")

0:	learn: 21.5624990	total: 1.24ms	remaining: 621ms
100:	learn: 1.1037679	total: 62.2ms	remaining: 246ms
200:	learn: 0.0564880	total: 184ms	remaining: 274ms
300:	learn: 0.0031476	total: 318ms	remaining: 210ms
400:	learn: 0.0002288	total: 398ms	remaining: 98.3ms
499:	learn: 0.0000032	total: 496ms	remaining: 0us
MAE: 33.71 minutes
R2: nan
Model and junction statistics saved.


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_regression.py:1266: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)


In [ ]:
import joblib

joblib.dump(model, "duration_model.pkl")


print("Model Saved")

Model Saved


In [ ]:
import joblib
import pandas as pd

def predict_duration(input_data: dict):
    """
    Runs inference on the saved CatBoost model to predict event duration.

    Args:
        input_data (dict): A dictionary containing event features. Keys should match
                           the feature names used during training.
                           'start_datetime' should be a string that can be parsed to datetime.
                           'junction' must be present.

    Returns:
        float: Predicted duration in minutes.
    """

    # Load the model and junction stats
    model = joblib.load('duration_model.pkl')
    junction_avg = joblib.load('junction_stats.pkl')

    # Convert input data to DataFrame
    df_inference = pd.DataFrame([input_data])

    # Recreate datetime features
    df_inference['start_datetime'] = pd.to_datetime(df_inference['start_datetime'])
    df_inference['hour'] = df_inference['start_datetime'].dt.hour
    df_inference['day_of_week'] = df_inference['start_datetime'].dt.dayofweek
    df_inference['month'] = df_inference['start_datetime'].dt.month
    df_inference['is_weekend'] = (df_inference['day_of_week'] >= 5).astype(int)

    # Recreate historical junction average
    df_inference['junction_avg_duration'] = df_inference['junction'].map(junction_avg).fillna(0) # Fill with 0 for unknown junctions

    # Define features used during training
    features = [
        "event_type",
        "event_cause",
        "requires_road_closure",
        "priority",
        "veh_type",
        "corridor",
        "police_station",
        "zone",
        "junction",
        "hour",
        "day_of_week",
        "month",
        "is_weekend",
        "junction_avg_duration"
    ]

    cat_features = [
        "event_type",
        "event_cause",
        "requires_road_closure",
        "priority",
        "veh_type",
        "corridor",
        "police_station",
        "zone",
        "junction"
    ]

    # Ensure all features are present
    for col in features:
        if col not in df_inference.columns:
            df_inference[col] = None # Add missing columns

    # Fill NaN values in categorical features with 'Unknown' for inference
    for col in cat_features:
        if col in df_inference.columns:
            df_inference[col] = df_inference[col].fillna('Unknown')

    # Select and reorder features to match training order
    X_inference = df_inference[features]

    # Make prediction
    prediction = model.predict(X_inference)
    return prediction[0]

print("Inference function 'predict_duration' defined.")

Inference function 'predict_duration' defined.


Here's an example of how to use the `predict_duration` function:

In [ ]:
# Example usage:
new_event_data = {
    'start_datetime': '2023-03-10 10:30:00',
    'event_type': 'Emergency',
    'event_cause': 'Accident',
    'requires_road_closure': 'No',
    'priority': 'High',
    'veh_type': 'Car',
    'corridor': 'Corridor B',
    'police_station': 'Station A',
    'zone': 'Zone 3',
    'junction': 'Junction 1'
}

predicted_duration = predict_duration(new_event_data)
print(f"Predicted duration for the new event: {predicted_duration:.2f} minutes")

# Example with an unknown junction
new_event_data_unknown_junction = {
    'start_datetime': '2023-03-10 10:30:00',
    'event_type': 'Emergency',
    'event_cause': 'Accident',
    'requires_road_closure': 'No',
    'priority': 'High',
    'veh_type': 'Car',
    'corridor': 'Corridor B',
    'police_station': 'Station A',
    'zone': 'Zone 3',
    'junction': 'Unknown Junction'
}

predicted_duration_unknown = predict_duration(new_event_data_unknown_junction)
print(f"Predicted duration for unknown junction: {predicted_duration_unknown:.2f} minutes")

Predicted duration for the new event: 48.03 minutes
Predicted duration for unknown junction: 49.69 minutes
